# 🎬 openGO_video — Colab GPU 影片生成伺服器

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MazingerSam/openGO_video/blob/main/notebooks/colab_opengovideo.ipynb)

本筆記本把 Colab 變成 **本機 Agent 專用的遠端 GPU 伺服器**：

- 🧠 使用開源模型：**AnimateDiff**（文字→影片）、**SVD-XT**（圖片→影片）
- 📡 透過 **Cloudflare Tunnel** 產生免註冊的公開網址
- 🤖 本機以 `python -m opengovideo` 送出工作，MP4 自動下載回 `outputs/`

> ⚠️ **執行前**：選單「執行階段 ➔ 變更執行階段類型」選 **L4 GPU**（推薦）或 **T4 GPU**。

## 步驟 0：檢查 GPU 環境

In [ ]:
!nvidia-smi

## 步驟 1：取得專案並安裝套件

從 GitHub 複製本專案（若已存在則更新），再安裝 Colab 端依賴。

In [ ]:
import os
import sys

REPO_URL = "https://github.com/MazingerSam/openGO_video.git"
WORKDIR = "/content/openGO_video"

if os.path.isdir(WORKDIR):
    !cd {WORKDIR} && git pull --ff-only -q
else:
    !git clone -q {REPO_URL} {WORKDIR}

!pip install -q -r {WORKDIR}/requirements-colab.txt

sys.path.insert(0, f"{WORKDIR}/src")
import opengovideo
print(f"openGO_video {opengovideo.__version__} 已就緒")

## 步驟 2：啟動橋接伺服器

以下儲存格會：
1. 預載兩個模型（約 1~2 分鐘，可略過讓第一次生成時再載入）
2. 啟動 FastAPI（`0.0.0.0:8000`）
3. 建立 Cloudflare Tunnel 並印出公開網址

看到 `🔗 公開網址：https://xxxx.trycloudflare.com` 後，回到本機執行 `python connect.py` 貼上網址即可。

> 若不想預載模型，把 `warmup(...)` 那行註解掉。

In [ ]:
from opengovideo.pipeline import warmup
from opengovideo.server import app, launch

# 可註解以節省啟動時間
warmup(app.state.store.pipeline, ["animatediff-v1.5", "svd-xt"])

launch(port=8000, tunnel=True)

## 補充：斷線後重啟

Colab 閒置斷線後，重新「執行階段 ➔ 全部執行」，再從最後一格複製新的 Tunnel 網址，在本機執行：

```powershell
python connect.py
```

即可繼續使用，不需要重新安裝任何東西。